# Introduction - Memory in Agentic Systems

## Progressive Memory Layering for Intelligent Agents

Agents that interact with users over multiple turns need to remember more than just the current query. Without memory, an agent cannot:
- Track user preferences and constraints
- Avoid repeating the same recommendations
- Learn patterns from past interactions
- Maintain context across conversation turns

This notebook demonstrates how layered memory—conversational history, structured state, embeddings, episodic history, and reflection—transforms a stateless LLM into a capable agentic system. Each layer serves a different role:

1. **Conversational Memory** records the dialogue (what was said)
2. **Structured State** captures the current context (what we're searching for)
3. **Embeddings** store semantic knowledge for retrieval (what we know)
4. **Episodic Memory** tracks past interactions (what we did)
5. **Reflection** captures learned insights (what we learned)

Together, these create a system where memory is a control primitive—actively shaping agent behavior at each turn.

## Memory Types: A Conceptual Overview

### Conversational Memory
**Purpose:** Record of the dialogue (what was said)  
**Example:** Full conversation history with user queries and assistant responses  
**Usage:** LLM uses context to understand references ("you mentioned earlier...")  
**Volatility:** Maintained within a session, can be archived between sessions

### Structured State Memory
**Purpose:** Maintains current constraints and context  
**Example:** Search filters (genres, ratings, year), user preferences  
**Usage:** Agent applies state filters to narrow results in real-time  
**Volatility:** Updated each turn, resets between sessions

### Long-Term Embedding Memory
**Purpose:** Semantic knowledge base for retrieval  
**Example:** User preferences, learned tastes, domain knowledge  
**Usage:** Retrieve relevant memories based on semantic similarity to current query  
**Volatility:** Persists across sessions, grows over time

### Episodic Memory
**Purpose:** History of past interactions and recommendations  
**Example:** Previously recommended movies, past search constraints  
**Usage:** Avoid recommending the same item twice, detect patterns  
**Volatility:** Accumulates across sessions, can be archived or forgotten

### Reflection Memory
**Purpose:** Learned insights and rules inferred from experience  
**Example:** "User prefers action films", "Tends to like movies after 2015"  
**Usage:** Refine reasoning and predictions for future queries  
**Volatility:** Persistent, explicit statements of patterns discovered

### Why Layered Memory?
No single memory type captures all needs. Combining them creates a system that:

- **Understands context** through dialogue (conversational)The key insight: **memory constrains the agent's decision space**, making it more predictable, controllable, and human-aligned.

- **Executes constraints** with precision (structured state)

- **Leverages knowledge** from past interactions (embeddings)- **Improves over time** through reflection (insights)
- **Avoids repetition** and learns patterns (episodic)

# 2. Setup & Environment

In [3]:
# Standard library imports
import sys
sys.path.append("../../agentic-ai")
import os
import pandas as pd
import numpy as np
from typing import List, Dict, Optional
from sklearn.metrics.pairwise import cosine_similarity

# Third-party imports
from langchain.chat_models import ChatOpenAI
from langchain.embeddings import OpenAIEmbeddings

# User-defined imports
from utils import utils

In [2]:
# Load environment variables
from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Initialize LLM and embeddings
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
embeddings = OpenAIEmbeddings()

C:\Users\cetar\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\langchain_core\_api\deprecation.py:119: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 0.3.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import ChatOpenAI`.
  warn_deprecated(
C:\Users\cetar\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\langchain_core\_api\deprecation.py:119: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 0.3.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_ope

In [6]:
# Load sample movie dataset for demonstration
df = pd.read_csv('../data/tmdb_5000_movies.csv')
df = df[['title','genres','release_date','vote_average','runtime','overview']].copy()
df['year'] = pd.to_datetime(df['release_date']).dt.year
df = df.dropna(subset=['year','runtime'])
print(f"Loaded {len(df)} movies")
print(f"\nSample movies:")
utils.print_html(df[['title', 'year', 'vote_average']].head(10))

Loaded 4800 movies

Sample movies:


title,year,vote_average
Avatar,2009.0,7.2
Pirates of the Caribbean: At World's End,2007.0,6.9
Spectre,2015.0,6.3
The Dark Knight Rises,2012.0,7.6
John Carter,2012.0,6.1
Spider-Man 3,2007.0,5.9
Tangled,2010.0,7.4
Avengers: Age of Ultron,2015.0,7.3
Harry Potter and the Half-Blood Prince,2009.0,7.4
Batman v Superman: Dawn of Justice,2016.0,5.7


# 3. Memory Implementation

### 3.1 Baseline: No Memory

**The Problem:** Without memory, an agent cannot maintain context across conversation turns.

Consider this interaction:
- User: "Recommend science fiction movies after 2020 with a rating above 7."
- Assistant: [Recommends 5 sci-fi films]
- User: "Only ones that are shorter than 2 hours."
- Assistant (without memory): "I don't know what you're looking for. Can you describe the genre?"

The second query should filter the first results, but without memory, the agent has lost all context.

Let's demonstrate this failure.

In [7]:
def stateless_recommend(query: str) -> str:
    """
    Generate a recommendation WITHOUT any memory.
    Each call is independent—no context is retained from prior queries.
    
    Args:
        query: The user's recommendation request.
    
    Returns:
        str: LLM-generated recommendations.
    """
    prompt = f"""
You are a movie recommendation assistant.
User query: {query}
Recommend 3-5 movies based on the request.
"""
    response = llm.invoke(prompt)
    return response.content

In [8]:
# Demonstrate the problem
print("=== FIRST QUERY ===")
query_1 = "Suggest science fiction movies after 2015 with a rating above 7."
print(f"User: {query_1}")
print("\nAssistant (stateless):")
response_1 = stateless_recommend(query_1)
utils.print_html(response_1)

=== FIRST QUERY ===
User: Suggest science fiction movies after 2015 with a rating above 7.

Assistant (stateless):


In [9]:
print("\n=== FOLLOW-UP QUERY (WITHOUT MEMORY) ===")
query_2 = "Only ones shorter than 2 hours."
print(f"User: {query_2}")
print("\nAssistant (stateless):")
response_2 = stateless_recommend(query_2)
utils.print_html(response_2)
print("\n❌ PROBLEM: The assistant has lost the original constraints!")
print("It doesn't remember 'science fiction after 2015 with rating > 7'")


=== FOLLOW-UP QUERY (WITHOUT MEMORY) ===
User: Only ones shorter than 2 hours.

Assistant (stateless):



❌ PROBLEM: The assistant has lost the original constraints!
It doesn't remember 'science fiction after 2015 with rating > 7'


### 3.2 Conversational Memory

**What is it?**  
Conversational memory (chat history) is a record of the actual dialogue turns—what the user said and what the assistant responded. It captures the exact flow of the conversation in natural language.

**Why use it?**  
The LLM has no innate knowledge of what was discussed in previous turns. By passing the full conversation history in each prompt, we give the LLM all the explicit context it needs to understand references like "did you mention X earlier?" or "based on what we talked about...".

**Format:**  
```
[
  {"role": "user", "content": "Recommend sci-fi after 2015"},
  {"role": "assistant", "content": "Here are some..."},
  {"role": "user", "content": "Only short ones"},
  ...
]
```

**Lifecycle:**  
- Initialize empty
- Append each user query and assistant response
- Pass full history to the LLM in each prompt
- Can be trimmed/summarized if it grows too large

Let's implement it:

In [10]:
# Initialize conversational memory (chat history)
conversation_history: List[Dict] = []

def add_conversation_turn(role: str, content: str) -> None:
    """
    Add a turn to the conversation history.
    
    Args:
        role: Either "user" or "assistant".
        content: The message text.
    """
    conversation_history.append({
        "role": role,
        "content": content
    })
    print(f"[{role.upper()}] {content[:60]}...")

def get_conversation_context() -> str:
    """
    Format the conversation history for inclusion in prompts.
    
    Returns:
        str: The conversation history as formatted text.
    """
    if not conversation_history:
        return "(No prior conversation)"
    
    context = "CONVERSATION HISTORY:\n"
    for turn in conversation_history:
        context += f"{turn['role'].upper()}: {turn['content']}\n"
    return context

# Demo: Simulate a multi-turn conversation
print("=== SIMULATING MULTI-TURN CONVERSATION ===\n")

# Turn 1
user_query_1 = "Recommend science fiction movies after 2015 with a rating above 7."
add_conversation_turn("user", user_query_1)
assistant_response_1 = "I recommend Avatar 2, Dune, and The Adam Project."
add_conversation_turn("assistant", assistant_response_1)

print(f"\n[Conversation has {len(conversation_history)} turns]\n")

# Turn 2
user_query_2 = "Only ones shorter than 2 hours."
add_conversation_turn("user", user_query_2)

print(f"\nConversation context for Turn 2:")
print(get_conversation_context())

=== SIMULATING MULTI-TURN CONVERSATION ===

[USER] Recommend science fiction movies after 2015 with a rating ab...
[ASSISTANT] I recommend Avatar 2, Dune, and The Adam Project....

[Conversation has 2 turns]

[USER] Only ones shorter than 2 hours....

Conversation context for Turn 2:
CONVERSATION HISTORY:
USER: Recommend science fiction movies after 2015 with a rating above 7.
ASSISTANT: I recommend Avatar 2, Dune, and The Adam Project.
USER: Only ones shorter than 2 hours.



In [11]:
# Show how conversational memory enables anaphora (reference to prior context)
def agent_with_conversation_context(query: str) -> str:
    """
    An agent that uses conversational memory to understand references to prior context.
    
    Args:
        query: The current user query.
    
    Returns:
        str: The agent's response.
    """
    prompt = f"""{get_conversation_context()}

USER: {query}

Based on the conversation history above, respond to the user's query. 
Understand that "Only ones shorter than 2 hours" refers back to the sci-fi movies recommended in the first turn.
"""
    response = llm.invoke(prompt)
    return response.content

In [12]:
print("With conversational memory, the agent understands references:")
response = agent_with_conversation_context(user_query_2)
print(f"\nAgent response (understands 'ones' = sci-fi movies from Turn 1):")
utils.print_html(response)

With conversational memory, the agent understands references:

Agent response (understands 'ones' = sci-fi movies from Turn 1):


### 3.3 Structured State Memory

**What is it?**  
Structured state captures the current search constraints and context. It's a simple data structure (dict or dataclass) that the agent updates as the user provides new constraints.

**Why use it?**  
Structured state makes constraints explicit and queryable. Instead of buried in natural language, filters are represented as structured fields that can be applied deterministically.

**Lifecycle:**  
- Initialize with empty/default values
- Update as user provides constraints
- Use to filter/search the database
- Persist through the conversation session

Let's implement it:

In [13]:
# Initialize structured state memory
def init_search_state() -> Dict:
    """
    Initialize the structured state memory.
    
    The state tracks:
    - genres: List of genre strings to filter by
    - min_year: Minimum release year
    - min_rating: Minimum vote average (0-10)
    - max_runtime: Maximum runtime in minutes
    - exclude_titles: Titles already recommended (to avoid repetition)
    
    Returns:
        Dict: Initial empty state.
    """
    return {
        'genres': [],
        'min_year': None,
        'min_rating': None,
        'max_runtime': None,
        'exclude_titles': []
    }

search_state = init_search_state()
print("Initial state:")
print(search_state)

Initial state:
{'genres': [], 'min_year': None, 'min_rating': None, 'max_runtime': None, 'exclude_titles': []}


In [14]:
def filter_movies(df: pd.DataFrame, state: Dict) -> pd.DataFrame:
    """
    Apply structured state filters to the movie dataset.
    
    Args:
        df: The movie dataframe.
        state: The search state containing filters.
    
    Returns:
        pd.DataFrame: Filtered movies matching the state constraints.
    """
    result = df.copy()
    
    if state['genres']:
        for g in state['genres']:
            result = result[result['genres'].str.contains(g, case=False, na=False)]
    if state['min_year']:
        result = result[result['year'] >= state['min_year']]
    if state['min_rating']:
        result = result[result['vote_average'] >= state['min_rating']]
    if state['max_runtime']:
        result = result[result['runtime'] <= state['max_runtime']]
    if state['exclude_titles']:
        result = result[~result['title'].isin(state['exclude_titles'])]
    
    return result

In [15]:
# Demo: Update state and filter
search_state['genres'] = ['Science Fiction']
search_state['min_year'] = 2015
search_state['min_rating'] = 7

filtered = filter_movies(df, search_state)
print("Filtered results (Sci-Fi after 2015, rating >= 7):")
utils.print_html(filtered[['title', 'year', 'vote_average']].head(5))
print(f"\nFound {len(filtered)} matching movies")

Filtered results (Sci-Fi after 2015, rating >= 7):


title,year,vote_average
Avengers: Age of Ultron,2015.0,7.3
Captain America: Civil War,2016.0,7.1
Mad Max: Fury Road,2015.0,7.2
Ant-Man,2015.0,7.0
The Martian,2015.0,7.6



Found 6 matching movies


In [16]:
# Now apply the second constraint
search_state['max_runtime'] = 120  # 2 hours

filtered = filter_movies(df, search_state)
print("After adding runtime constraint (< 2 hours):")
utils.print_html(filtered[['title', 'year', 'vote_average', 'runtime']].head(5))
print(f"\nFound {len(filtered)} matching movies")
print("\n✅ SUCCESS: State is preserved! We remembered the original constraints.")

After adding runtime constraint (< 2 hours):


title,year,vote_average,runtime
Mad Max: Fury Road,2015.0,7.2,120.0
Ant-Man,2015.0,7.0,117.0
Ex Machina,2015.0,7.6,108.0



Found 3 matching movies

✅ SUCCESS: State is preserved! We remembered the original constraints.


### 3.4 Long-Term Embedding Memory

**What is it?**  
Long-term memory stores text embeddings—semantic representations of knowledge. When the user mentions a preference, we encode it and store it. Later, when a new query arrives, we retrieve the most similar memories using cosine similarity.

**Why use it?**  
Embeddings capture *semantic meaning* beyond exact keyword matching. "I like action-packed adventures" and "I enjoy thrilling films" are conceptually similar, even though they use different words. Embeddings capture this.

**Lifecycle:**  
- Convert text to embedding via OpenAI API (or other model)
- Store embeddings in memory (list or vector DB)
- Retrieve by computing cosine similarity to query embedding
- Persist across sessions

Let's implement it:

In [17]:
# Initialize long-term embedding memory
long_term_memory: List[Dict] = []

def add_memory(content: str) -> None:
    """
    Add a memory to long-term storage.
    
    The memory is converted to an embedding and stored alongside the original text
    so we can later retrieve by semantic similarity.
    
    Args:
        content: Text of the memory (e.g., user preference).
    """
    emb = embeddings.embed_query(content)
    long_term_memory.append({
        'content': content,
        'embedding': np.array(emb)
    })
    print(f"Memory added: {content}")

def retrieve_memory(query: str, top_k: int = 3) -> List[str]:
    """
    Retrieve the most similar memories to the query.
    
    Computes cosine similarity between the query embedding and all stored memories,
    then returns the top-k most similar.
    
    Args:
        query: The query text.
        top_k: Number of memories to retrieve.
    
    Returns:
        List[str]: The top-k most similar memories.
    """
    if not long_term_memory:
        return []
    
    q_emb = np.array(embeddings.embed_query(query)).reshape(1, -1)
    mem_embs = np.vstack([m['embedding'] for m in long_term_memory])
    sims = cosine_similarity(q_emb, mem_embs)[0]
    top_idx = sims.argsort()[-top_k:][::-1]
    
    return [long_term_memory[i]['content'] for i in top_idx]

# Demo: Add memories
add_memory("User prefers fast-paced science fiction films.")
add_memory("User enjoyed movies from the 2015.")
add_memory("User likes movies with ratings above 7.0.")

Memory added: User prefers fast-paced science fiction films.
Memory added: User enjoyed movies from the 2015.
Memory added: User likes movies with ratings above 7.0.


In [19]:
# Retrieve memories related to a new query
query = "Recommend exciting sci-fi"
retrieved = retrieve_memory(query, top_k=2)

print(f"\nQuery: {query}")
print(f"\nTop-2 retrieved memories:")
for i, mem in enumerate(retrieved, 1):
    print(f"{i}. {mem}")
print("\n✅ Semantic retrieval: Similar concepts are retrieved even with different wording.")


Query: Recommend exciting sci-fi

Top-2 retrieved memories:
1. User prefers fast-paced science fiction films.
2. User likes movies with ratings above 7.0.

✅ Semantic retrieval: Similar concepts are retrieved even with different wording.


### 3.5 Episodic Memory

**What is it?**  
Episodic memory is a journal of past interactions. Each "episode" captures what filters were applied and what was recommended. This allows the agent to:
- Avoid recommending the same item repeatedly
- Detect patterns in what the user views across sessions
- Understand the history of the conversation

**Why use it?**  
Without episodic memory, the agent might recommend the same movie three times across different conversation turns, frustrating the user.

**Structure:**  
Each episode contains:
- The search filters applied
- The titles recommended
- (Optionally) timestamp, user reaction, etc.

Let's implement it:

In [21]:
# Initialize episodic memory
episodic_memory: List[Dict] = []

def add_episode(filters: Dict, titles: List[str]) -> None:
    """
    Record an episode (interaction) in episodic memory.
    
    Args:
        filters: The search filters applied (copy of search_state).
        titles: The titles recommended in this episode.
    """
    episodic_memory.append({
        'filters': filters.copy(),
        'titles': titles
    })
    print(f"Episode recorded: {len(titles)} titles, state={filters}")

def get_previous_titles() -> List[str]:
    """
    Get all titles that have been recommended in past episodes.
    
    Returns:
        List[str]: Unique titles from all past episodes.
    """
    titles = []
    for ep in episodic_memory:
        titles.extend(ep['titles'])
    return list(set(titles))

In [22]:
# Demo: Get recommendations and add to episodic memory
search_state = init_search_state()
search_state['genres'] = ['Science Fiction']
search_state['min_year'] = 2015
filtered = filter_movies(df, search_state)
recommended_titles = filtered.head(3)['title'].tolist()

utils.print_html(f"First recommendations: {recommended_titles}")
add_episode(search_state, recommended_titles)

Episode recorded: 3 titles, state={'genres': ['Science Fiction'], 'min_year': 2015, 'min_rating': None, 'max_runtime': None, 'exclude_titles': []}


In [24]:
# Later: Make new recommendations but exclude prior titles
search_state['min_rating'] = 7
founded_after_filter = filter_movies(df, search_state)

# Exclude titles we've already recommended
previous_titles = get_previous_titles()
search_state['exclude_titles'] = previous_titles

filtered_new = filter_movies(df, search_state)
new_recommended = filtered_new.head(3)['title'].tolist()

utils.print_html(f"\nPrevious titles (from episodic memory): {previous_titles}")
utils.print_html(f"New recommendations (excluding prior): {new_recommended}")
print("\n✅ Episodic memory: We never recommend the same title twice!")


✅ Episodic memory: We never recommend the same title twice!


### 3.6 Reflection Memory

**What is it?**  
Reflection memory stores *learned insights*—high-level patterns or rules inferred from past interactions. Examples:
- "User avoids movies longer than 2 hours"
- "User has a strong preference for action films"
- "User tends to rate older movies higher"

**Why use it?**  
Reflection makes the agent "wiser" over time. Instead of reacting mechanically to each query, it applies learned wisdom to improve recommendations.

**How is it created?**  
Reflections can be:
- Explicitly added by observing patterns (e.g., "I notice you always filter by year > 2015")
- Generated by the LLM analyzing past episodes
- Inferred from implicit user feedback (ratings, views, etc.)

Let's implement it:

In [25]:
# Initialize reflection memory
reflection_memory: List[str] = []

def add_reflection(insight: str) -> None:
    """
    Add a learned insight to reflection memory.
    
    Args:
        insight: A text description of a pattern or rule learned.
    """
    reflection_memory.append(insight)
    print(f"Reflection added: {insight}")

def get_reflections() -> str:
    """
    Get all reflections as a formatted string for inclusion in prompts.
    
    Returns:
        str: All reflections joined with newlines.
    """
    return '\n'.join(reflection_memory) if reflection_memory else "(No reflections yet)"

In [28]:
# Demo: Add reflections
add_reflection("User strongly prefers science fiction and action genres.")
add_reflection("User avoids movies longer than 2 hours.")
add_reflection("User tends to favor movies with ratings above 7.0.")

print("\nAll reflections:")
utils.print_html(get_reflections())

Reflection added: User strongly prefers science fiction and action genres.
Reflection added: User avoids movies longer than 2 hours.
Reflection added: User tends to favor movies with ratings above 7.0.

All reflections:


In [29]:
# Reflections can inform the agent's reasoning
print("\n✅ Reflection memory: The agent can now apply learned wisdom in its recommendations.")
print("\nExample: When a user asks 'Recommend a movie', the agent can do better because it knows:")
utils.print_html(get_reflections())


✅ Reflection memory: The agent can now apply learned wisdom in its recommendations.

Example: When a user asks 'Recommend a movie', the agent can do better because it knows:


# 4. Memory-Augmented Agent: Orchestration

**How it works:**  
The memory-augmented agent combines all four memory types into a single context that it passes to the LLM. Each memory layer provides different information:

1. **Structured State** → What constraints apply right now?
2. **Long-Term Embeddings** → What similar preferences has the user expressed?
3. **Episodic Memory** → What has already been recommended?
4. **Reflection** → What patterns have we learned?

The prompt builds a narrative context from these layers, allowing the LLM to make decisions that respect all of them.

In [30]:
def memory_agent(user_query: str, verbose: bool = True) -> str:
    """
    A memory-augmented agent that uses all four memory types to generate recommendations.
    
    Args:
        user_query: The user's request.
        verbose: If True, print debug output showing what memories are being used.
    
    Returns:
        str: The agent's response.
    """
    # Retrieve from all memory types
    retrieved_memories = retrieve_memory(user_query, top_k=3)
    reflections = get_reflections()
    previous_titles = get_previous_titles()
    
    # Build context
    state_summary = {
        "genres": search_state.get("genres"),
        "min_year": search_state.get("min_year"),
        "min_rating": search_state.get("min_rating"),
        "max_runtime": search_state.get("max_runtime"),
    }
    
    if verbose:
        print("\n" + "="*60)
        print("MEMORY DEBUG OUTPUT")
        print("="*60)
        print("\nStructured State Memory (Current Constraints):")
        for key, val in state_summary.items():
            if val:
                print(f"  {key}: {val}")
        
        print("\nLong-Term Embedding Memory (Retrieved Preferences):")
        for i, mem in enumerate(retrieved_memories, 1):
            print(f"  {i}. {mem}")
        
        print("\nEpisodic Memory (Prior Recommendations):")
        print(f"  Excluded titles: {previous_titles}")
        
        print("\nReflection Memory (Learned Insights):")
        print(f"  {reflections}")
        print("="*60 + "\n")
    
    # Build the prompt
    prompt = f"""
You are a movie recommendation assistant with access to the user's memory.

CURRENT SEARCH STATE (what we're filtering by now):
{state_summary}

USER PREFERENCES (from long-term memory):
{retrieved_memories}

PRIOR RECOMMENDATIONS (avoid repeating these):
{previous_titles}

LEARNED INSIGHTS (apply these lessons):
{reflections}

USER QUERY:
{user_query}

Provide movie recommendations that respect ALL constraints.
"""
    
    response = llm.invoke(prompt)
    return response.content

In [32]:
# Demo: Use the memory-augmented agent
print("=== MEMORY-AUGMENTED AGENT ===")
response = memory_agent("Recommend a movie to watch this weekend.")
print("\nAgent Response:")
utils.print_html(response)

=== MEMORY-AUGMENTED AGENT ===

MEMORY DEBUG OUTPUT

Structured State Memory (Current Constraints):
  genres: ['Science Fiction']
  min_year: 2015
  min_rating: 7

Long-Term Embedding Memory (Retrieved Preferences):
  1. User likes movies with ratings above 7.0.
  2. User prefers fast-paced science fiction films.
  3. User enjoyed movies from the 2015.

Episodic Memory (Prior Recommendations):
  Excluded titles: ['Captain America: Civil War', 'Jurassic World', 'Avengers: Age of Ultron']

Reflection Memory (Learned Insights):
  User strongly prefers science fiction and action genres.
User avoids movies longer than 2 hours.
User tends to favor movies with ratings above 7.0.
User strongly prefers science fiction and action genres.
User avoids movies longer than 2 hours.
User tends to favor movies with ratings above 7.0.
User strongly prefers science fiction and action genres.
User avoids movies longer than 2 hours.
User tends to favor movies with ratings above 7.0.


Agent Response:


In [33]:
# Compare with stateless agent
print("\n=== COMPARISON: STATELESS VS MEMORY-AUGMENTED ===")
print("\nStateless Agent (no memory):")
response_stateless = stateless_recommend("Recommend a movie to watch this weekend.")
utils.print_html(response_stateless)

print("\n---")
print("\nMemory-Augmented Agent (with all 5 memory types):")
response_augmented = memory_agent("Recommend a movie to watch this weekend.", verbose=False)
utils.print_html(response_augmented)

print("\n✅ The memory-augmented agent produces more contextual, personalized recommendations.")


=== COMPARISON: STATELESS VS MEMORY-AUGMENTED ===

Stateless Agent (no memory):



---

Memory-Augmented Agent (with all 5 memory types):



✅ The memory-augmented agent produces more contextual, personalized recommendations.


# 5. Takeaways & Design Principles

### Key Insights

#### Memory as a Control Primitive
Memory is not just storage—it's a **controller** that shapes agent behavior. By carefully designing what and how the agent remembers, we can:
- Make agents more predictable and controllable
- Reduce unwanted behavior (repetition, inconsistency)
- Encode human values and preferences systematically
- Debug and audit agent decisions ("Why did it do that?" → Look at memory)

#### When to Use Each Memory Type

| Memory Type | Best For | Example |
|---|---|---|
| **Conversational** | Multi-turn dialogue understanding | Understanding anaphora ("you mentioned...") |
| **Structured State** | Constrained search/filtering tasks | Movie recommendations with filters |
| **Embeddings** | Semantic knowledge retention | "User prefers fast-paced films" |
| **Episodic** | Multi-turn interactions | Prior recommendations, interaction history |
| **Reflection** | Learning patterns | "User tends to like movies after 2015" |

#### Extension Ideas

**Persistence to Disk**
- Save memory structures to JSON/database
- Load at session start to provide continuity across restarts

**Memory Summarization**
- Use LLM to summarize episodic history ("In the last 10 conversations, user recommended X...")
- Reduces context window usage

**TTL on Memories**
- Memories expire after N days
- Useful for capturing trends but forgetting stale preferences

**Hierarchical Memory**
- Short-term (current conversation)
- Medium-term (last week)
- Long-term (lifetime)
- Different retrieval strategies per tier

#### Final Thought

The most effective agentic systems don't just respond to the current input—they **integrate information across time** using carefully designed memory. Memory is how agents achieve consistency, personalization, and effectiveness.